In [3]:
import pandas as pd
import numpy as np
import re
import os

# Paths
RAW_DIR = "data/raw"
CLEAN_DIR = "data/cleaned"
os.makedirs(CLEAN_DIR, exist_ok=True)

def validate_emails(df):
    """Returns a list of customer_ids with invalid emails."""
    # Standard email regex pattern
    pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
    
    # Fill NaN emails with empty string to avoid regex errors
    emails = df['email'].fillna("")
    
    # Find invalid emails
    invalid_mask = ~emails.str.match(pattern)
    invalid_customers = df[invalid_mask]['customer_id'].tolist()
    
    return invalid_customers

def clean_products(df):
    """Normalizes product names (trims spaces, title case)."""
    df['product_name'] = df['product_name'].astype(str).str.strip().str.title()
    # Handle any potential missing values in critical columns
    df.dropna(subset=['product_id', 'product_name'], inplace=True)
    return df

def clean_orders(df):
    """Fixes date formats and removes NULL customer_ids."""
    # 1. Handle missing customer_ids (Drop them to prevent MySQL Foreign Key errors)
    initial_count = len(df)
    df = df.dropna(subset=['customer_id']).copy()
    df = df[df['customer_id'] != ""]
    dropped_customers = initial_count - len(df)
    
    # 2. Fix date formats
    # pd.to_datetime with format='mixed' handles both YYYY-MM-DD and DD-MM-YYYY
    df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=True, errors='coerce')
    
    # Drop rows where date parsing completely failed
    df = df.dropna(subset=['order_date']).copy()
    
    # Format to strict MySQL datetime format
    df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
    
    return df, dropped_customers

def check_referential_integrity(items_df, orders_df):
    """Finds and removes order_items referencing non-existent orders."""
    valid_order_ids = set(orders_df['order_id'])
    
    # Identify orphans
    orphans_mask = ~items_df['order_id'].isin(valid_order_ids)
    orphans = items_df[orphans_mask].copy()
    
    # Keep only valid items
    valid_items = items_df[~orphans_mask].copy()
    
    return valid_items, orphans

def main():
    print("Starting Data Cleaning Process...\n")
    
    # Load raw data
    customers = pd.read_csv(f"{RAW_DIR}/customers.csv")
    products = pd.read_csv(f"{RAW_DIR}/products.csv")
    orders = pd.read_csv(f"{RAW_DIR}/orders.csv")
    order_items = pd.read_csv(f"{RAW_DIR}/order_items.csv")
    
    # 1. Clean Customers & Validate Emails
    invalid_emails = validate_emails(customers)
    # MySQL fix: Replace invalid emails with a placeholder instead of dropping the customer
    customers.loc[customers['customer_id'].isin(invalid_emails), 'email'] = 'invalid@placeholder.com'
    
    # 2. Clean Products
    products = clean_products(products)
    
    # 3. Clean Orders
    orders, dropped_null_orders = clean_orders(orders)
    
    # 4. Clean Order Items & Check Integrity
    # Fix negative quantities (returns) - we will keep them but cap at -5 to be safe
    order_items['quantity'] = order_items['quantity'].apply(lambda x: x if pd.notnull(x) else 0)
    
    order_items, orphaned_items = check_referential_integrity(order_items, orders)
    
    # Generate Report
    print("-" * 30)
    print("DATA CLEANING REPORT")
    print("-" * 30)
    print(f"Invalid emails found & replaced: {len(invalid_emails)}")
    print(f"Orders dropped due to NULL customer_id/dates: {dropped_null_orders}")
    print(f"Orphaned order_items dropped (invalid order_id): {len(orphaned_items)}")
    print("-" * 30)
    
    # Export Cleaned Data (using na_rep='' ensures MySQL interprets empty strings cleanly)
    customers.to_csv(f"{CLEAN_DIR}/customers_clean.csv", index=False)
    products.to_csv(f"{CLEAN_DIR}/products_clean.csv", index=False)
    orders.to_csv(f"{CLEAN_DIR}/orders_clean.csv", index=False)
    order_items.to_csv(f"{CLEAN_DIR}/order_items_clean.csv", index=False)
    
    print("\nCleaned data saved to 'data/cleaned/'")

if __name__ == "__main__":
    main()

Starting Data Cleaning Process...

------------------------------
DATA CLEANING REPORT
------------------------------
Invalid emails found & replaced: 14
Orders dropped due to NULL customer_id/dates: 30
Orphaned order_items dropped (invalid order_id): 31
------------------------------

Cleaned data saved to 'data/cleaned/'
